In [1]:
# ============================================================
# STAGE 4 - WEB APPLICATION INTEGRATION
# CELL 1: PROJECT CHECK
# ============================================================

from pathlib import Path
import os

PROJECT_ROOT = Path(
    r"E:\Pyhton code\Deep Learning\CNN\Face Recognition Model"
)

DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "labeled wild face dataset 13000 images"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "face_recognition_resnet50.pth"
)

APP_DIR = PROJECT_ROOT / "app"

APP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 60)
print("STAGE 4 - PROJECT CHECK")
print("=" * 60)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nDataset exists:")
print(DATASET_DIR.exists())

print("\nModel exists:")
print(MODEL_PATH.exists())

print("\nModel:")
print(MODEL_PATH)

print("\nApp directory:")
print(APP_DIR)

print("\n✓ Stage 4 environment ready.")

STAGE 4 - PROJECT CHECK

Project root:
E:\Pyhton code\Deep Learning\CNN\Face Recognition Model

Dataset exists:
True

Model exists:
True

Model:
E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\models\face_recognition_resnet50.pth

App directory:
E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\app

✓ Stage 4 environment ready.


In [2]:
# ============================================================
# STAGE 4 - CELL 2
# CREATE FACE RECOGNITION ENGINE
# ============================================================

engine_code = r'''
# ============================================================
# FACE RECOGNITION ENGINE
# YuNet + OpenCV Enhancement + ResNet-50
# ============================================================

from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(__file__).resolve().parent.parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "face_recognition_resnet50.pth"
)

YUNET_PATH = (
    PROJECT_ROOT
    / "models"
    / "face_detection_yunet_2023mar.onnx"
)


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# SETTINGS
# ============================================================

IMAGE_SIZE = 224

# Recognition threshold.
#
# We deliberately do NOT use a very low threshold.
# If the model is not confident, it should say Unknown.

RECOGNITION_THRESHOLD = 0.50

DETECTION_THRESHOLD = 0.50

TOP_K = 5


# ============================================================
# MODEL ARCHITECTURE
# EXACT ARCHITECTURE USED DURING TRAINING
# ============================================================

class FaceRecognitionModel(nn.Module):

    def __init__(
        self,
        num_classes,
        embedding_dim=512
    ):

        super().__init__()

        # ----------------------------------------------------
        # ResNet-50 backbone
        # ----------------------------------------------------

        backbone = resnet50(
            weights=None
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone


        # ----------------------------------------------------
        # 2048 -> 512 embedding
        # ----------------------------------------------------

        self.embedding = nn.Sequential(

            nn.Linear(
                2048,
                embedding_dim
            ),

            nn.BatchNorm1d(
                embedding_dim
            ),

            nn.ReLU()
        )


        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.classifier = nn.Linear(
            embedding_dim,
            num_classes
        )


    def forward(self, x):

        features = self.backbone(x)

        embedding = self.embedding(
            features
        )

        # L2 normalization

        embedding = embedding / (
            torch.norm(
                embedding,
                p=2,
                dim=1,
                keepdim=True
            ) + 1e-10
        )

        logits = self.classifier(
            embedding
        )

        return logits, embedding


# ============================================================
# LOAD MODEL
# ============================================================

def load_model():

    checkpoint = torch.load(
        MODEL_PATH,
        map_location=DEVICE,
        weights_only=False
    )

    selected_people = checkpoint[
        "selected_people"
    ]

    embedding_dim = checkpoint[
        "embedding_dim"
    ]

    num_classes = len(
        selected_people
    )

    model = FaceRecognitionModel(
        num_classes=num_classes,
        embedding_dim=embedding_dim
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model = model.to(DEVICE)

    model.eval()

    return (
        model,
        selected_people,
        checkpoint
    )


# ============================================================
# IMAGE TRANSFORMATION
# MUST MATCH TRAINING
# ============================================================

transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# YUNET FACE DETECTOR
# ============================================================

def create_detector():

    if not YUNET_PATH.exists():

        raise FileNotFoundError(
            f"YuNet model not found:\n{YUNET_PATH}"
        )

    detector = cv2.FaceDetectorYN.create(

        str(YUNET_PATH),

        "",

        (320, 320),

        DETECTION_THRESHOLD,

        0.3,

        5000
    )

    return detector


# ============================================================
# ENHANCEMENT
# LEVEL 1 OPENCV ENHANCEMENT
# ============================================================

def enhance_face(face):

    if face is None:
        return None

    # --------------------------------------------------------
    # Resize before enhancement
    # --------------------------------------------------------

    height, width = face.shape[:2]

    if width < 300 or height < 300:

        scale = max(
            300 / width,
            300 / height
        )

        new_width = int(
            width * scale
        )

        new_height = int(
            height * scale
        )

        face = cv2.resize(
            face,
            (new_width, new_height),
            interpolation=cv2.INTER_CUBIC
        )


    # --------------------------------------------------------
    # CLAHE enhancement
    # --------------------------------------------------------

    lab = cv2.cvtColor(
        face,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = (
        cv2.split(lab)
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_channel = clahe.apply(
        l_channel
    )

    lab = cv2.merge(
        (
            l_channel,
            a_channel,
            b_channel
        )
    )

    enhanced = cv2.cvtColor(
        lab,
        cv2.COLOR_LAB2BGR
    )


    # --------------------------------------------------------
    # Mild sharpening
    # --------------------------------------------------------

    blurred = cv2.GaussianBlur(
        enhanced,
        (0, 0),
        1.0
    )

    enhanced = cv2.addWeighted(
        enhanced,
        1.25,
        blurred,
        -0.25,
        0
    )

    return enhanced


# ============================================================
# DETECT FACES
# ============================================================

def detect_faces(
    image,
    detector
):

    if image is None:
        return []


    height, width = image.shape[:2]

    detector.setInputSize(
        (width, height)
    )


    _, detections = detector.detect(
        image
    )


    if detections is None:
        return []


    results = []


    for detection in detections:

        x, y, w, h = (
            detection[:4]
            .astype(int)
        )

        confidence = float(
            detection[14]
        )


        # Clamp coordinates

        x1 = max(
            0,
            x
        )

        y1 = max(
            0,
            y
        )

        x2 = min(
            width,
            x + w
        )

        y2 = min(
            height,
            y + h
        )


        if x2 <= x1 or y2 <= y1:
            continue


        face = image[
            y1:y2,
            x1:x2
        ].copy()


        results.append({

            "bbox": (
                x1,
                y1,
                x2 - x1,
                y2 - y1
            ),

            "confidence": confidence,

            "face": face
        })


    return results


# ============================================================
# PREPARE FACE
# ============================================================

def prepare_face(face):

    enhanced = enhance_face(
        face
    )

    rgb = cv2.cvtColor(
        enhanced,
        cv2.COLOR_BGR2RGB
    )

    pil_image = Image.fromarray(
        rgb
    )

    tensor = transform(
        pil_image
    )

    tensor = tensor.unsqueeze(
        0
    )

    return (
        tensor.to(DEVICE),
        enhanced
    )


# ============================================================
# RECOGNIZE ONE FACE
# ============================================================

def recognize_face(
    face,
    model,
    selected_people
):

    input_tensor, enhanced = (
        prepare_face(face)
    )


    with torch.no_grad():

        logits, embedding = model(
            input_tensor
        )


    probabilities = torch.softmax(
        logits,
        dim=1
    )[0]


    probabilities_np = (
        probabilities
        .cpu()
        .numpy()
    )


    sorted_indices = np.argsort(
        probabilities_np
    )[::-1]


    top_predictions = []


    for index in sorted_indices[:TOP_K]:

        top_predictions.append({

            "name":
                selected_people[index],

            "confidence":
                float(
                    probabilities_np[index]
                )
        })


    best_index = sorted_indices[0]

    best_confidence = float(
        probabilities_np[
            best_index
        ]
    )

    best_name = selected_people[
        best_index
    ]


    # --------------------------------------------------------
    # UNKNOWN DECISION
    # --------------------------------------------------------

    if (
        best_confidence
        < RECOGNITION_THRESHOLD
    ):

        identity = "Unknown"

    else:

        identity = best_name


    return {

        "identity": identity,

        "confidence":
            best_confidence,

        "embedding":
            embedding[0]
            .cpu()
            .numpy(),

        "enhanced_face":
            enhanced,

        "top_predictions":
            top_predictions
    }


# ============================================================
# PROCESS COMPLETE IMAGE
# ============================================================

def process_image(
    image,
    model,
    detector,
    selected_people
):

    faces = detect_faces(
        image,
        detector
    )


    results = []


    for face_data in faces:

        recognition = recognize_face(

            face_data["face"],

            model,

            selected_people
        )


        result = {

            "bbox":
                face_data["bbox"],

            "detection_confidence":
                face_data["confidence"],

            **recognition
        }


        results.append(
            result
        )


    return results
'''

engine_path = APP_DIR / "face_engine.py"

with open(
    engine_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        engine_code
    )


print("=" * 60)
print("FACE ENGINE CREATED")
print("=" * 60)

print("\nFile:")
print(engine_path)

print("\n✓ YuNet detector")
print("✓ OpenCV enhancement")
print("✓ Exact ResNet-50 architecture")
print("✓ 512-D embedding")
print("✓ Recognition")
print("✓ Unknown detection")
print("\n✓ Engine ready.")

FACE ENGINE CREATED

File:
E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\app\face_engine.py

✓ YuNet detector
✓ OpenCV enhancement
✓ Exact ResNet-50 architecture
✓ 512-D embedding
✓ Recognition
✓ Unknown detection

✓ Engine ready.


In [3]:
# ============================================================
# STAGE 4 - CELL 3
# CREATE STREAMLIT APPLICATION
# ============================================================

app_code = r'''
# ============================================================
# AI FACE RECOGNITION SYSTEM
# Stage 4
# ============================================================

import sys
from pathlib import Path

import cv2
import numpy as np
import streamlit as st


# ============================================================
# PATH
# ============================================================

APP_DIR = Path(
    __file__
).resolve().parent

PROJECT_ROOT = APP_DIR.parent

sys.path.insert(
    0,
    str(APP_DIR)
)


from face_engine import (
    load_model,
    create_detector,
    process_image
)


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(

    page_title="AI Face Recognition",

    page_icon="👁️",

    layout="wide"
)


# ============================================================
# TITLE
# ============================================================

st.title(
    "👁️ AI Face Recognition System"
)

st.caption(
    "YuNet + OpenCV Enhancement + "
    "Custom ResNet-50 Face Recognition"
)


# ============================================================
# LOAD MODEL
# ============================================================

@st.cache_resource
def initialize_system():

    model, selected_people, checkpoint = (
        load_model()
    )

    detector = create_detector()

    return (
        model,
        detector,
        selected_people,
        checkpoint
    )


try:

    (
        model,
        detector,
        selected_people,
        checkpoint
    ) = initialize_system()

except Exception as e:

    st.error(
        f"Could not initialize AI system:\n\n{e}"
    )

    st.stop()


# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header(
    "⚙️ System Information"
)

st.sidebar.write(
    f"Classes: {len(selected_people)}"
)

st.sidebar.write(
    "Embedding: 512-D"
)

st.sidebar.write(
    "Detector: YuNet"
)

st.sidebar.write(
    "Backbone: ResNet-50"
)


# ============================================================
# INPUT MODE
# ============================================================

mode = st.radio(

    "Select input source",

    [
        "📷 Image",
        "🎥 Video",
        "🔴 Live Camera"
    ],

    horizontal=True
)


# ============================================================
# DRAW RESULT
# ============================================================

def draw_results(
    image,
    results
):

    output = image.copy()


    for result in results:

        x, y, w, h = (
            result["bbox"]
        )

        identity = result[
            "identity"
        ]

        confidence = result[
            "confidence"
        ]


        if identity == "Unknown":

            label = (
                f"Unknown "
                f"({confidence * 100:.1f}%)"
            )

        else:

            label = (
                f"{identity} "
                f"({confidence * 100:.1f}%)"
            )


        # ----------------------------------------------------
        # Bounding box
        # ----------------------------------------------------

        cv2.rectangle(

            output,

            (x, y),

            (x + w, y + h),

            (0, 255, 0),

            2
        )


        # ----------------------------------------------------
        # Label
        # ----------------------------------------------------

        cv2.putText(

            output,

            label,

            (x, max(25, y - 10)),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.65,

            (0, 255, 0),

            2
        )


    return output


# ============================================================
# IMAGE MODE
# ============================================================

if mode == "📷 Image":

    st.subheader(
        "Upload an image"
    )


    uploaded_file = st.file_uploader(

        "Choose an image",

        type=[
            "jpg",
            "jpeg",
            "png",
            "webp"
        ]
    )


    if uploaded_file is not None:

        file_bytes = np.asarray(
            bytearray(
                uploaded_file.read()
            ),
            dtype=np.uint8
        )


        image = cv2.imdecode(
            file_bytes,
            cv2.IMREAD_COLOR
        )


        if image is None:

            st.error(
                "Could not read image."
            )

            st.stop()


        # ----------------------------------------------------
        # Process
        # ----------------------------------------------------

        with st.spinner(
            "Detecting and recognizing faces..."
        ):

            results = process_image(

                image,

                model,

                detector,

                selected_people
            )


        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        result_image = draw_results(
            image,
            results
        )


        result_rgb = cv2.cvtColor(

            result_image,

            cv2.COLOR_BGR2RGB
        )


        st.image(

            result_rgb,

            caption="Recognition Result",

            use_container_width=True
        )


        # ----------------------------------------------------
        # Results
        # ----------------------------------------------------

        st.subheader(
            "Recognition Results"
        )


        if len(results) == 0:

            st.warning(
                "No reliable face detected."
            )

        else:

            for i, result in enumerate(
                results,
                start=1
            ):

                st.write(
                    f"### Face {i}"
                )

                st.write(
                    f"Identity: "
                    f"**{result['identity']}**"
                )

                st.write(
                    f"Recognition confidence: "
                    f"**{result['confidence'] * 100:.2f}%**"
                )

                st.write(
                    f"Detection confidence: "
                    f"**{result['detection_confidence'] * 100:.2f}%**"
                )


                with st.expander(
                    "Top predictions"
                ):

                    for prediction in (
                        result[
                            "top_predictions"
                        ]
                    ):

                        st.write(

                            f"{prediction['name']} "
                            f"— "
                            f"{prediction['confidence'] * 100:.2f}%"
                        )


# ============================================================
# VIDEO MODE
# ============================================================

elif mode == "🎥 Video":

    st.subheader(
        "Upload a video"
    )


    uploaded_video = st.file_uploader(

        "Choose a video",

        type=[
            "mp4",
            "avi",
            "mov",
            "mkv"
        ]
    )


    if uploaded_video is not None:

        video_path = (
            PROJECT_ROOT
            / "app"
            / "_uploaded_video.mp4"
        )


        with open(
            video_path,
            "wb"
        ) as f:

            f.write(
                uploaded_video.read()
            )


        st.video(
            str(video_path)
        )


        st.info(
            "Video upload is working. "
            "Frame-by-frame recognition "
            "and tracking will be added "
            "in Stage 5."
        )


# ============================================================
# LIVE CAMERA MODE
# ============================================================

elif mode == "🔴 Live Camera":

    st.subheader(
        "Live camera"
    )


    st.info(
        "Use the camera button below "
        "to provide live frames."
    )


    camera_image = st.camera_input(
        "Take a picture from your camera"
    )


    if camera_image is not None:

        file_bytes = np.asarray(

            bytearray(
                camera_image.read()
            ),

            dtype=np.uint8
        )


        image = cv2.imdecode(

            file_bytes,

            cv2.IMREAD_COLOR
        )


        results = process_image(

            image,

            model,

            detector,

            selected_people
        )


        result_image = draw_results(

            image,

            results
        )


        result_rgb = cv2.cvtColor(

            result_image,

            cv2.COLOR_BGR2RGB
        )


        st.image(

            result_rgb,

            caption="Camera Recognition",

            use_container_width=True
        )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "Stage 4 — AI Face Recognition System"
)
'''

app_path = APP_DIR / "app.py"

with open(
    app_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        app_code
    )


print("=" * 60)
print("STREAMLIT APP CREATED")
print("=" * 60)

print("\nFile:")
print(app_path)

print("\n✓ Image mode")
print("✓ Video upload mode")
print("✓ Camera mode")
print("✓ Face detection")
print("✓ Face enhancement")
print("✓ Face recognition")

print("\n✓ Stage 4 application created.")

STREAMLIT APP CREATED

File:
E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\app\app.py

✓ Image mode
✓ Video upload mode
✓ Camera mode
✓ Face detection
✓ Face enhancement
✓ Face recognition

✓ Stage 4 application created.


In [5]:
# ============================================================
# STAGE 4 - CELL 4
# VERIFY APPLICATION FILES
# ============================================================

print("=" * 60)
print("STAGE 4 FILE CHECK")
print("=" * 60)


files_to_check = [

    APP_DIR / "app.py",

    APP_DIR / "face_engine.py",

    MODEL_PATH,

    PROJECT_ROOT
    / "models"
    / "face_detection_yunet_2023mar.onnx"
]


for path in files_to_check:

    print(
        f"\n{path}"
    )

    print(
        "Exists:",
        path.exists()
    )

    if path.exists():

        print(
            "Size:",
            path.stat().st_size,
            "bytes"
        )


print("\n✓ File check completed.")

STAGE 4 FILE CHECK

E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\app\app.py
Exists: True
Size: 9756 bytes

E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\app\face_engine.py
Exists: True
Size: 11508 bytes

E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\models\face_recognition_resnet50.pth
Exists: True
Size: 98590491 bytes

E:\Pyhton code\Deep Learning\CNN\Face Recognition Model\models\face_detection_yunet_2023mar.onnx
Exists: True
Size: 232589 bytes

✓ File check completed.
